# How to Build a RAG System with Claude + Oracle AI Database Vector Search

This notebook demonstrates how to build an enterprise-grade Retrieval-Augmented Generation (RAG) pipeline using:
- **Claude (Anthropic)** for generation
- **Oracle AI Database Vector Search** for retrieval

We go beyond a basic demo to showcase Oracle's native vector capabilities:
- **`VECTOR_DISTANCE`** – real similarity scoring computed by the database
- **Hybrid search** – combining vector similarity with relational SQL filters
- **Vector indexing** – creating HNSW indexes for scalable approximate nearest neighbor search
- **Strongly typed vectors** – `VECTOR(1024, FLOAT32)` for precise memory control

You will learn how to:
1. Set up the environment and install dependencies
2. Connect to Oracle AI Database
3. Create a vector table with relational metadata columns
4. Generate embeddings and ingest documents
5. Create a vector index for scalable search
6. Perform similarity search with real distance scores
7. Run hybrid queries (vector + relational filters)
8. Use retrieved context to answer questions with Claude

### Step 1: Install dependencies

We install the required Python libraries:
- `oracledb` – Oracle Database driver
- `anthropic` – Claude API client
- `python-dotenv` – load secrets from `.env`
- `pandas`, `numpy`, `matplotlib` – data inspection and visualization


In [1]:
!pip install -U oracledb anthropic python-dotenv pandas numpy sentence-transformers -q

### Prerequisites

This notebook assumes you already have access to an Oracle AI Database instance.

You can use either:

- **Oracle Autonomous Database (recommended for cloud demos / production)**
  - Provision an Autonomous Database on OCI
  - Configure wallet + secure connection

- **Oracle AI Database Free (local)**
  - Run locally via Docker / Podman  
  - Official image: `container-registry.oracle.com/database/free:latest`

Before continuing, make sure:
- Your Oracle AI DB is running
- You can connect using `oracledb`
- Your connection details are set in `.env`

Docs:
- Oracle AI Database Vector Search: https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/

### Step 2: Load environment variables

Create a `.env` file with the following values:

- ANTHROPIC_API_KEY=...
- ORACLE_USER=...
- ORACLE_PASSWORD=...
- ORACLE_DSN=localhost:1521/FREEPDB1


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

print("ORACLE_USER:", os.getenv("ORACLE_USER"))
print("ORACLE_DSN:", os.getenv("ORACLE_DSN"))
print("ANTHROPIC_API_KEY loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))


ORACLE_USER: RAG_USER
ORACLE_DSN: localhost:1521/FREEPDB1
ANTHROPIC_API_KEY loaded: True


### Step 3: Connect to Oracle AI Database

You can connect either to:

- **Oracle Autonomous Database (recommended for production & cloud demos)**
- **Oracle AI Database Free running locally (Docker/Podman) for local testing**

Both options use the same Python API (`oracledb`).  
You only need to change the connection configuration.


> For Autonomous DB, make sure your `ORACLE_DSN` points to the Autonomous service
> and that you have configured the wallet or secure connection parameters.

> For example:
> ORACLE_DSN=adb_high


In [3]:
import oracledb
import os

# Works for both:
# - Autonomous DB (with wallet + DSN)
# - Local Oracle AI DB Free (Docker/Podman)
conn = oracledb.connect(
    user=os.getenv("ORACLE_USER"),
    password=os.getenv("ORACLE_PASSWORD"),
    dsn=os.getenv("ORACLE_DSN"),
)

cursor = conn.cursor()
print("Connected to Oracle AI Database")


Connected to Oracle AI Database


### Step 4: Create vector table

We create a table with:
- `id` – auto-generated primary key
- `content` – the document text (CLOB)
- `category` – relational metadata column for hybrid filtering
- `embedding VECTOR(1024, FLOAT32)` – Oracle native vector type with explicit precision

Using `VECTOR(1024, FLOAT32)` instead of just `VECTOR(1024)` gives Oracle precise control over memory layout and ensures consistent distance calculations.

The `category` column enables **hybrid search** — combining vector similarity with standard SQL `WHERE` filters in a single query. This is one of Oracle's key advantages over standalone vector databases.

In [ ]:
cursor.execute("""
BEGIN
  EXECUTE IMMEDIATE 'DROP TABLE documents PURGE';
EXCEPTION
  WHEN OTHERS THEN NULL;
END;
""")

cursor.execute("""
CREATE TABLE documents (
    id NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    content CLOB,
    category VARCHAR2(100),
    embedding VECTOR(1024, FLOAT32)
)
""")

conn.commit()
print("Vector table created with VECTOR(1024, FLOAT32) and category column")

### Step 5: Prepare documents

We use a small in-memory dataset with **category labels** for each document.

The `category` field is a standard relational column that enables hybrid search — filtering by metadata while performing vector similarity search in the same query.

In real applications, documents could come from PDFs, web pages, product catalogs, or internal knowledge bases, each tagged with relevant metadata.

In [ ]:
docs = [
    {"content": "Oracle AI Database provides native vector search for semantic retrieval across enterprise data.", "category": "oracle"},
    {"content": "Vector Search enables similarity queries over embeddings stored directly in Oracle tables.", "category": "oracle"},
    {"content": "RAG systems combine retrieval with LLMs to reduce hallucinations and improve factual grounding.", "category": "rag"},
    {"content": "Claude is used for generation and reasoning, while embeddings can be produced by a separate model.", "category": "llm"},
    {"content": "Ingesting documents into a vector store enables semantic search over unstructured content.", "category": "rag"},
    {"content": "Top-k retrieval returns the most relevant chunks based on vector distance metrics.", "category": "rag"},
    {"content": "Chunking strategies (size, overlap) significantly impact retrieval quality in RAG systems.", "category": "rag"},
    {"content": "Oracle AI Database integrates vector search with transactional and analytical workloads.", "category": "oracle"},
    {"content": "Python applications can connect to Oracle using the oracledb driver for vector operations.", "category": "oracle"},
    {"content": "Grounding LLM answers in retrieved context improves reliability and auditability.", "category": "llm"},
    {"content": "RAG pipelines typically include ingestion, indexing, retrieval, and generation steps.", "category": "rag"},
    {"content": "Vector indexes can accelerate similarity search for large corpora.", "category": "oracle"},
    {"content": "Developers should monitor similarity score distributions to tune retrieval thresholds.", "category": "rag"},
    {"content": "Evaluating retrieval quality is crucial before optimizing LLM prompts.", "category": "rag"},
    {"content": "Domain-specific corpora produce better grounded answers than generic text.", "category": "rag"},
    {"content": "Product documentation and FAQs are common sources for RAG knowledge bases.", "category": "rag"},
    {"content": "Chunk overlap can improve recall but may increase redundancy.", "category": "rag"},
    {"content": "Embedding model choice affects semantic recall and precision.", "category": "embedding"},
    {"content": "Vector search can power semantic search, Q&A, and recommendations.", "category": "oracle"},
    {"content": "Oracle Vector Search supports cosine and L2 distance metrics for similarity.", "category": "oracle"},
    {"content": "To build a RAG system with Oracle and Claude, first ingest documents into Oracle AI Database as vectors.", "category": "tutorial"},
    {"content": "Each document is embedded using an embedding model and stored in a VECTOR column.", "category": "tutorial"},
    {"content": "At query time, the user question is embedded and used for similarity search in Oracle Vector Search.", "category": "tutorial"},
    {"content": "The top-k retrieved documents are passed as context to Claude for grounded generation.", "category": "tutorial"},
    {"content": "Claude generates the final answer based only on the retrieved context.", "category": "tutorial"},
    {"content": "RAG systems combine retrieval from a vector database with LLM generation to reduce hallucinations.", "category": "rag"},
    {"content": "Oracle AI Database provides native VECTOR types and similarity search for RAG pipelines.", "category": "oracle"},
    {"content": "Python applications can orchestrate ingestion, retrieval, and LLM calls in a RAG workflow.", "category": "tutorial"},
]
print(f"Documents loaded: {len(docs)} across categories: {sorted(set(d['category'] for d in docs))}")

### Step 6: Generate embeddings

We convert each document into a numeric vector using a local embedding model 
from SentenceTransformers.

This avoids relying on closed embedding APIs and makes the cookbook easier 
to reproduce for everyone.

Claude is used only for generation (LLM), not for embeddings.


In [7]:
from sentence_transformers import SentenceTransformer
import logging
import os
import warnings

logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

embedder = SentenceTransformer("intfloat/e5-large-v2")

def embed(text: str):
    return embedder.encode(text, normalize_embeddings=True, show_progress_bar=False).tolist()

v = embed("Oracle Vector Search with Claude")
print("Embedding model loaded. Vector dimension =", len(v))


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 582.99it/s, Materializing param=pooler.dense.weight]                               


Embedding model loaded. Vector dimension = 1024


### Step 7: Ingest documents into Oracle AI Database

We generate embeddings for each document and insert them — along with their category metadata — into Oracle AI Database.

Oracle expects vectors in binary format, so we convert Python lists into `array("f")` before inserting.

In [ ]:
from array import array

for d in docs:
    emb = embed(d["content"])

    assert len(emb) == 1024, f"Vector dim mismatch: {len(emb)}"

    vec = array("f", emb)

    cursor.execute(
        "INSERT INTO documents (content, category, embedding) VALUES (:1, :2, :3)",
        [d["content"], d["category"], vec]
    )

conn.commit()
print(f"Ingested {len(docs)} documents with categories into Oracle AI Database")

### Step 8: Create a vector index

For our 28-document demo, Oracle can perform exact nearest neighbor (KNN) search efficiently. But at enterprise scale (millions of rows), you need **Approximate Nearest Neighbor (ANN) indexes**.

Oracle supports HNSW (Hierarchical Navigable Small World) vector indexes for fast, scalable similarity search:

```sql
CREATE VECTOR INDEX ... ORGANIZATION INMEMORY NEIGHBOR GRAPH
```

Key parameters:
- **DISTANCE COSINE** — matches the metric used in queries
- **TARGET ACCURACY 95** — trade-off between speed and recall (higher = more accurate but slower)

Even for this small dataset, we create the index to demonstrate the syntax and show that Oracle is production-ready.

> **Note:** On Oracle Free containers with limited memory, index creation may be skipped (ORA-51962). The notebook will continue to work using exact KNN search. On Autonomous DB or production instances with sufficient `vector_memory_size`, the index will be created successfully.

In [ ]:
try:
    cursor.execute("""
    CREATE VECTOR INDEX doc_vec_idx ON documents (embedding)
    ORGANIZATION INMEMORY NEIGHBOR GRAPH
    DISTANCE COSINE
    WITH TARGET ACCURACY 95
    """)
    conn.commit()
    print("HNSW vector index created (COSINE distance, target accuracy 95%)")
except Exception as e:
    print(f"Note: Vector index creation skipped — {e}")
    print("This is expected on Oracle Free containers with limited memory.")
    print("On Autonomous DB or production instances, the index will be created successfully.")
    print("Queries will still work using exact KNN search.")

### Step 9: Perform similarity search with real distance scores

We use Oracle's `VECTOR_DISTANCE` function to retrieve the top-K most similar documents **with their actual cosine distance** computed by the database.

This is the **retrieval** step of RAG. Unlike mocked scores, these distances are computed in real-time by Oracle's vector engine:

```sql
SELECT content, VECTOR_DISTANCE(embedding, :query_vector, COSINE) AS distance
FROM documents ORDER BY distance FETCH FIRST 3 ROWS ONLY
```

- **Cosine distance** ranges from 0 (identical) to 2 (opposite)
- **Cosine similarity** = 1 - cosine distance
- The `ORDER BY distance` clause leverages the vector index for fast retrieval

In [ ]:
query = "How do I build a RAG system with Oracle and Claude?"
q_emb = embed(query)
q_vec = array("f", q_emb)

# Use VECTOR_DISTANCE to get real cosine distance scores from Oracle
cursor.execute("""
    SELECT content,
           VECTOR_DISTANCE(embedding, :1, COSINE) AS distance
    FROM documents
    ORDER BY distance
    FETCH FIRST 3 ROWS ONLY
""", [q_vec])

rows = cursor.fetchall()

retrieved = []
for r in rows:
    text = r[0].read() if hasattr(r[0], "read") else r[0]
    distance = r[1]
    retrieved.append({
        "content": text,
        "distance": round(distance, 4),
        "similarity": round(1 - distance, 4)
    })

import pandas as pd
df_results = pd.DataFrame(retrieved)
print("Top-3 results ranked by Oracle VECTOR_DISTANCE (COSINE):\n")
print(df_results.to_string(index=False))

# Build context string for Claude generation
context = "\n---\n".join([r["content"] for r in retrieved])

### Step 10: Hybrid search — relational filters + vector similarity

One of Oracle's greatest advantages over standalone vector databases is the ability to combine **vector similarity search** with **standard SQL filters** in a single, ACID-compliant query.

This means you can ask: *"Find documents similar to X, but only in the 'tutorial' category."*

No post-filtering, no separate index lookups — Oracle optimizes the entire query as one operation.

In [ ]:
# Hybrid search: vector similarity + relational WHERE clause in one query
cursor.execute("""
    SELECT content, category,
           VECTOR_DISTANCE(embedding, :1, COSINE) AS distance
    FROM documents
    WHERE category = 'tutorial'
    ORDER BY distance
    FETCH FIRST 3 ROWS ONLY
""", [q_vec])

rows = cursor.fetchall()

print("Hybrid Search: Vector Similarity + WHERE category = 'tutorial'\n")
print(f"{'Category':<12} {'Distance':<10} Content")
print("-" * 90)
for r in rows:
    text = r[0].read() if hasattr(r[0], "read") else r[0]
    category = r[1]
    distance = r[2]
    print(f"{category:<12} {distance:<10.4f} {text[:65]}...")

### Step 11: Generate a grounded answer with Claude

We pass the retrieved context to Claude and instruct it to answer **only using the provided documents**.

This reduces hallucinations and ensures answers are grounded in your data.

In [10]:
from anthropic import Anthropic
import os

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))


### Model selection

Recommended models (check [docs.anthropic.com](https://docs.anthropic.com) for latest):

- **Claude Sonnet 4.5**: `claude-sonnet-4-5-20250929` (best balance of quality and speed)
- **Claude Haiku 4.5**: `claude-haiku-4-5-20251001` (fastest, good for demos)
- **Claude Opus 4.5**: `claude-opus-4-5-20251101` (highest quality)

In [ ]:
resp = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": f"""
You are a helpful assistant.

Context:
{context}

Question:
{query}

Answer clearly and concisely based only on the context.
"""
        }
    ]
)

print("Final answer:\n", resp.content[0].text)

### Inspect retrieval quality

The similarity scores below are **real values** computed by Oracle's `VECTOR_DISTANCE` function — not mocked or hardcoded.

This helps you:
- Debug retrieval quality
- Tune chunking strategies and top-k thresholds
- Evaluate embedding model performance
- Compare distance metrics (COSINE vs L2)

In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame(retrieved)

fig, ax = plt.subplots(figsize=(10, 3))
bars = ax.barh(range(len(df)), df["similarity"], color="#c74634")
ax.set_yticks(range(len(df)))
ax.set_yticklabels([c[:70] + "..." if len(c) > 70 else c for c in df["content"]], fontsize=9)
ax.set_xlabel("Cosine Similarity (1 − distance)")
ax.set_title("Retrieval Quality: Real Scores from Oracle VECTOR_DISTANCE")
ax.invert_yaxis()

for i, (d, s) in enumerate(zip(df["distance"], df["similarity"])):
    ax.text(s + 0.01, i, f"dist={d:.4f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

### Step 12: Cleanup resources

In [13]:
try:
    cursor.close()
    conn.close()
    print("Oracle DB connection closed.")
except Exception as e:
    print("Cleanup error:", e)

Oracle DB connection closed.
